In [1]:
!pip install chromadb sentence-transformers pandas numpy

In [2]:
import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully!")

Libraries imported successfully!


In [12]:
# This creates a small sample dataset first. After it displays the table, we’ll move to ChromaDB.
food_data = [...]
df = pd.DataFrame(food_data)
# so we can text chroma db embedding this (aka converting to vector) and storing it

In [27]:
import pandas as pd

df = pd.read_csv("food_recommendation_dataset.csv")
df.head()

,name,cuisine,calories,ingredients,cooking_method,taste,diet_tags,meal_type,description
0,Paneer Tikka,Indian,320,"paneer, yogurt, spices, bell pepper, onion",grilled,"spicy, smoky","vegetarian, high-protein",appetizer,A spicy grilled Indian dish made with marinate...
1,Chana Masala,Indian,380,"chickpeas, tomato, onion, ginger, garlic, spices",stewed,"spicy, savory","vegan, vegetarian, high-fiber",main,"A hearty chickpea curry cooked with tomatoes, ..."
2,Vegetable Biryani,Indian,520,"basmati rice, mixed vegetables, spices, yogurt...",steamed,"aromatic, spicy",vegetarian,main,"A fragrant rice dish layered with vegetables, ..."
3,Dal Tadka,Indian,300,"lentils, garlic, cumin, onion, tomato, ghee",simmered,"savory, comforting","vegetarian, high-protein",main,A comforting lentil dish finished with a flavo...
4,Tandoori Chicken,Indian,410,"chicken, yogurt, chili, turmeric, garam masala",grilled,"spicy, smoky",high-protein,main,Chicken marinated in spiced yogurt and grilled...


In [16]:
# Create a persistent ChromaDB client
# Create (or get) a collection
# collection is like a folder where all the data is stored

import chromadb

client = chromadb.PersistentClient(path="./food_chroma_db")

# Delete the old collection if it exists
try:
    client.delete_collection("food_collection")
except:
    pass

# Create a fresh collection
collection = client.get_or_create_collection("food_collection")

print("Fresh collection ready!")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Fresh collection ready!


In [17]:
# Clear existing data (optional, useful if re-running)
for i, row in df.iterrows():
    document = (
        f"{row['name']}. {row['description']} "
        f"Ingredients: {row['ingredients']}. "
        f"Taste: {row['taste']}. "
        f"Cooking method: {row['cooking_method']}."
    )

    collection.add(
        ids=[str(i)],
        documents=[document],
        metadatas=[{
            "name": row["name"],
            "cuisine": row["cuisine"],
            "calories": int(row["calories"]),
            "cooking_method": row["cooking_method"],
            "taste": row["taste"],
            "diet_tags": row["diet_tags"],
            "meal_type": row["meal_type"]
        }]
    )

print(f"Successfully added {len(df)} foods to ChromaDB!")

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Successfully added 69 foods to ChromaDB!


In [18]:
# basic similarity search: where vector of stored data vs vector of promt is compared - matching ones/ least distance are matched

query = "I want something spicy and grilled"

results = collection.query(
    query_texts=[query],
    n_results=3
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}")
    print("Food:", results["metadatas"][0][i]["name"])
    print("Cuisine:", results["metadatas"][0][i]["cuisine"])
    print("Calories:", results["metadatas"][0][i]["calories"])
    print("Description:", doc)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Result 1
Food: Paneer Tikka
Cuisine: Indian
Calories: 320
Description: Paneer Tikka. A spicy grilled Indian dish made with marinated paneer and vegetables. Ingredients: paneer, yogurt, spices, bell pepper, onion. Taste: spicy, smoky. Cooking method: grilled.

Result 2
Food: Chicken Souvlaki
Cuisine: Greek
Calories: 390
Description: Chicken Souvlaki. Grilled chicken skewers flavored with lemon, garlic, and oregano. Ingredients: chicken, lemon, oregano, garlic, pita. Taste: zesty, savory. Cooking method: grilled.

Result 3
Food: Tandoori Chicken
Cuisine: Indian
Calories: 410
Description: Tandoori Chicken. Chicken marinated in spiced yogurt and grilled until smoky and tender. Ingredients: chicken, yogurt, chili, turmeric, garam masala. Taste: spicy, smoky. Cooking method: grilled.


In [19]:
# metadata filtering: attribites (example: calories, cooking method, etc) - before doing similarity search
query = "I want something spicy"

results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"cuisine": "Indian"}
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}")
    print("Food:", results["metadatas"][0][i]["name"])
    print("Cuisine:", results["metadatas"][0][i]["cuisine"])
    print("Calories:", results["metadatas"][0][i]["calories"])
    print("Description:", doc)


Result 1
Food: Tandoori Chicken
Cuisine: Indian
Calories: 410
Description: Tandoori Chicken. Chicken marinated in spiced yogurt and grilled until smoky and tender. Ingredients: chicken, yogurt, chili, turmeric, garam masala. Taste: spicy, smoky. Cooking method: grilled.

Result 2
Food: Paneer Tikka
Cuisine: Indian
Calories: 320
Description: Paneer Tikka. A spicy grilled Indian dish made with marinated paneer and vegetables. Ingredients: paneer, yogurt, spices, bell pepper, onion. Taste: spicy, smoky. Cooking method: grilled.

Result 3
Food: Chana Masala
Cuisine: Indian
Calories: 380
Description: Chana Masala. A hearty chickpea curry cooked with tomatoes, onions, and warming spices. Ingredients: chickpeas, tomato, onion, ginger, garlic, spices. Taste: spicy, savory. Cooking method: stewed.


In [8]:
query = "I want something light and fresh"

results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"calories": {"$lt": 300}}
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}")
    print("Food:", results["metadatas"][0][i]["name"])
    print("Cuisine:", results["metadatas"][0][i]["cuisine"])
    print("Calories:", results["metadatas"][0][i]["calories"])
    print("Description:", doc)


Result 1
Food: Greek Salad
Cuisine: Greek
Calories: 180
Description: Greek Salad. A refreshing salad with vegetables, feta cheese, and olives. Ingredients: tomato, cucumber, feta cheese, olives, onion. Taste: fresh, tangy. Cooking method: fresh.

Result 2
Food: Sushi Roll
Cuisine: Japanese
Calories: 250
Description: Sushi Roll. A Japanese dish made with seasoned rice, seaweed, and fillings. Ingredients: rice, seaweed, fish, cucumber, avocado. Taste: fresh, savory. Cooking method: raw.


In [20]:
query = input("🍽️ What kind of food are you looking for? ")
max_calories = int(input("Maximum calories? "))

results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"calories": {"$lte": max_calories}}
)

print("\nTop Recommendations:\n")

for i, doc in enumerate(results["documents"][0]):
    metadata = results["metadatas"][0][i]

    print(f"Result {i+1}")
    print(f"Food: {metadata['name']}")
    print(f"Cuisine: {metadata['cuisine']}")
    print(f"Calories: {metadata['calories']}")
    print(f"Description: {doc}")
    print("-" * 50)

🍽️ What kind of food are you looking for? something light
Maximum calories? 100

Top Recommendations:

Result 1
Food: Miso Soup
Cuisine: Japanese
Calories: 90
Description: Miso Soup. A light Japanese soup with miso, tofu, seaweed, and scallions. Ingredients: miso paste, tofu, seaweed, scallions, dashi. Taste: umami, light. Cooking method: simmered.
--------------------------------------------------


In [21]:
query = input("🍽️ What kind of food are you looking for? ")

cuisine = input("🌍 Cuisine filter? Press Enter to skip: ")
max_calories = input("🔥 Maximum calories? Press Enter to skip: ")
meal_type = input("🍱 Meal type? Press Enter to skip: ")

where_filter = {}

if cuisine.strip():
    where_filter["cuisine"] = cuisine.strip()

if max_calories.strip():
    where_filter["calories"] = {"$lte": int(max_calories)}

if meal_type.strip():
    where_filter["meal_type"] = meal_type.strip()

if where_filter:
    results = collection.query(
        query_texts=[query],
        n_results=5,
        where=where_filter
    )
else:
    results = collection.query(
        query_texts=[query],
        n_results=5
    )

print("\nTop Recommendations:\n")

if not results["documents"][0]:
    print("No matching foods found. Try changing your filters.")
else:
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]

        print(f"Result {i+1}")
        print(f"Food: {metadata['name']}")
        print(f"Cuisine: {metadata['cuisine']}")
        print(f"Calories: {metadata['calories']}")
        print(f"Meal Type: {metadata['meal_type']}")
        print(f"Cooking Method: {metadata['cooking_method']}")
        print(f"Taste: {metadata['taste']}")
        print(f"Description: {doc}")
        print("-" * 50)

🍽️ What kind of food are you looking for? something light
🌍 Cuisine filter? Press Enter to skip: 
🔥 Maximum calories? Press Enter to skip: 
🍱 Meal type? Press Enter to skip: 

Top Recommendations:

Result 1
Food: Onigiri
Cuisine: Japanese
Calories: 210
Meal Type: snack
Cooking Method: assembled
Taste: savory, simple
Description: Onigiri. A rice ball wrapped with seaweed and filled with savory ingredients. Ingredients: rice, seaweed, tuna, sesame. Taste: savory, simple. Cooking method: assembled.
--------------------------------------------------
Result 2
Food: Mango Sticky Rice
Cuisine: Thai
Calories: 420
Meal Type: dessert
Cooking Method: steamed
Taste: sweet, creamy
Description: Mango Sticky Rice. A sweet Thai dessert with coconut sticky rice and ripe mango. Ingredients: sticky rice, mango, coconut milk, sugar. Taste: sweet, creamy. Cooking method: steamed.
--------------------------------------------------
Result 3
Food: Chicken Tagine
Cuisine: Moroccan
Calories: 480
Meal Type: main

In [22]:
new_food_id = "custom_1"

new_document = (
    "Grilled Veggie Wrap. A light and healthy wrap filled with grilled vegetables, "
    "hummus, lettuce, and whole wheat tortilla. "
    "Ingredients: whole wheat tortilla, zucchini, bell pepper, hummus, lettuce. "
    "Taste: smoky, fresh, savory. "
    "Cooking method: grilled."
)

new_metadata = {
    "name": "Grilled Veggie Wrap",
    "cuisine": "Healthy",
    "calories": 310,
    "cooking_method": "grilled",
    "taste": "smoky, fresh, savory",
    "diet_tags": "vegetarian, healthy",
    "meal_type": "lunch"
}

collection.add(
    ids=[new_food_id],
    documents=[new_document],
    metadatas=[new_metadata]
)

print("New food item added successfully!")

New food item added successfully!


In [ ]:
#| Letter | Meaning    | In our project                |
#| ------ | ---------- | ----------------------------- |
#| **C**  | **Create** | Add a new food item           |
#| **R**  | **Read**   | Search or retrieve food items |
#| **U**  | **Update** | Modify an existing food item  |
#| **D**  | **Delete** | Remove a food item            |

# The CRUD operations we've been doing modify the data stored in ChromaDB, NOT the original CSV file


In [24]:
results = collection.query(
    query_texts=["healthy grilled vegetarian lunch"],
    n_results=5
)

for i, doc in enumerate(results["documents"][0]):
    metadata = results["metadatas"][0][i]
    print(f"\nResult {i+1}")
    print("Food:", metadata["name"])
    print("Calories:", metadata["calories"])
    print("Description:", doc)


Result 1
Food: Grilled Veggie Wrap
Calories: 310
Description: Grilled Veggie Wrap. A light and healthy wrap filled with grilled vegetables, hummus, lettuce, and whole wheat tortilla. Ingredients: whole wheat tortilla, zucchini, bell pepper, hummus, lettuce. Taste: smoky, fresh, savory. Cooking method: grilled.

Result 2
Food: Veggie Burger
Calories: 480
Description: Veggie Burger. A hearty burger made with a plant-based vegetable patty. Ingredients: bean patty, bun, lettuce, tomato, sauce. Taste: savory, hearty. Cooking method: grilled.

Result 3
Food: Grilled Salmon
Calories: 390
Description: Grilled Salmon. Salmon grilled with lemon and herbs, often served with vegetables. Ingredients: salmon, lemon, herbs, olive oil, asparagus. Taste: savory, fresh. Cooking method: grilled.

Result 4
Food: Turkey Sandwich
Calories: 420
Description: Turkey Sandwich. A simple sandwich with turkey, vegetables, and mustard. Ingredients: whole wheat bread, turkey, lettuce, tomato, mustard. Taste: savory

In [25]:
collection.update(
    ids=["custom_1"],
    documents=[
        "Grilled Veggie Wrap. A healthy whole wheat wrap with grilled vegetables and hummus. Ingredients: whole wheat tortilla, zucchini, bell pepper, hummus, lettuce. Taste: smoky, fresh, savory. Cooking method: grilled."
    ],
    metadatas=[{
        "name": "Grilled Veggie Wrap",
        "cuisine": "Healthy",
        "calories": 280,   # Updated
        "cooking_method": "grilled",
        "taste": "smoky, fresh, savory",
        "diet_tags": "vegetarian, healthy",
        "meal_type": "lunch"
    }]
)

print("Food updated successfully!")

Failed to send telemetry event CollectionUpdateEvent: capture() takes 1 positional argument but 3 were given


Food updated successfully!


In [26]:
collection.delete(ids=["custom_1"])

print("Food deleted successfully!")

Failed to send telemetry event CollectionDeleteEvent: capture() takes 1 positional argument but 3 were given


Food deleted successfully!


In [28]:
def rag_food_chatbot(user_query, n_results=3):
    results = collection.query(
        query_texts=[user_query],
        n_results=n_results
    )

    if not results["documents"][0]:
        return "Sorry, I couldn't find any matching foods."

    response = f"Based on your request: '{user_query}', here are my recommendations:\n\n"

    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]

        response += f"{i+1}. {metadata['name']}\n"
        response += f"   Cuisine: {metadata['cuisine']}\n"
        response += f"   Calories: {metadata['calories']}\n"
        response += f"   Meal Type: {metadata['meal_type']}\n"
        response += f"   Why it matches: {doc}\n\n"

    response += "You can refine your request by adding cuisine, calories, taste, or meal type."

    return response

In [29]:
print(rag_food_chatbot("I want a light vegetarian lunch"))

Based on your request: 'I want a light vegetarian lunch', here are my recommendations:

1. Veggie Burger
   Cuisine: American
   Calories: 480
   Meal Type: main
   Why it matches: Veggie Burger. A hearty burger made with a plant-based vegetable patty. Ingredients: bean patty, bun, lettuce, tomato, sauce. Taste: savory, hearty. Cooking method: grilled.

2. Turkey Sandwich
   Cuisine: American
   Calories: 420
   Meal Type: lunch
   Why it matches: Turkey Sandwich. A simple sandwich with turkey, vegetables, and mustard. Ingredients: whole wheat bread, turkey, lettuce, tomato, mustard. Taste: savory, light. Cooking method: assembled.

3. Couscous Salad
   Cuisine: Moroccan
   Calories: 290
   Meal Type: salad
   Why it matches: Couscous Salad. A light couscous salad with chickpeas, vegetables, and herbs. Ingredients: couscous, chickpeas, cucumber, tomato, herbs. Taste: fresh, herby. Cooking method: assembled.

You can refine your request by adding cuisine, calories, taste, or meal type.


In [38]:
# RAG CHATBOT
def rag_food_chatbot_with_filters(user_query, cuisine=None, max_calories=None, meal_type=None, diet_tag=None, n_results=5):
    conditions = []

    if cuisine:
        conditions.append({"cuisine": cuisine})

    if max_calories:
        conditions.append({"calories": {"$lte": int(max_calories)}})

    if meal_type:
        conditions.append({"meal_type": meal_type})

    if len(conditions) == 1:
        where_filter = conditions[0]
    elif len(conditions) > 1:
        where_filter = {"$and": conditions}
    else:
        where_filter = None

    query_args = {
        "query_texts": [user_query],
        "n_results": 20
    }

    if where_filter:
        query_args["where"] = where_filter

    results = collection.query(**query_args)

    recommendations = []

    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]

        if diet_tag:
            if diet_tag.lower() not in metadata["diet_tags"].lower():
                continue

        recommendations.append((metadata, doc))

        if len(recommendations) == n_results:
            break

    if not recommendations:
        return "Sorry, I couldn't find matching foods. Try relaxing your filters."

    response = f"Based on your request: '{user_query}', here are my recommendations:\n\n"

    for i, (metadata, doc) in enumerate(recommendations):
        response += f"{i+1}. {metadata['name']}\n"
        response += f"   Cuisine: {metadata['cuisine']}\n"
        response += f"   Calories: {metadata['calories']}\n"
        response += f"   Meal Type: {metadata['meal_type']}\n"
        response += f"   Diet Tags: {metadata['diet_tags']}\n"
        response += f"   Why it matches: {doc}\n\n"

    return response


In [39]:
print(
    rag_food_chatbot_with_filters(
        user_query="I want a light vegetarian lunch",
        max_calories=350,
        diet_tag="vegetarian",
        n_results=5
    )
)

Based on your request: 'I want a light vegetarian lunch', here are my recommendations:

1. Couscous Salad
   Cuisine: Moroccan
   Calories: 290
   Meal Type: salad
   Diet Tags: vegetarian, low-calorie
   Why it matches: Couscous Salad. A light couscous salad with chickpeas, vegetables, and herbs. Ingredients: couscous, chickpeas, cucumber, tomato, herbs. Taste: fresh, herby. Cooking method: assembled.

2. Papaya Salad
   Cuisine: Thai
   Calories: 160
   Meal Type: salad
   Diet Tags: vegan, vegetarian, low-calorie
   Why it matches: Papaya Salad. A crunchy Thai salad with green papaya, lime, chili, and peanuts. Ingredients: green papaya, lime, chili, peanuts, tomato. Taste: spicy, tangy. Cooking method: fresh.

3. Zucchini Noodles
   Cuisine: Healthy
   Calories: 220
   Meal Type: main
   Diet Tags: vegetarian, low-carb
   Why it matches: Zucchini Noodles. Spiralized zucchini cooked with tomato sauce, garlic, and basil. Ingredients: zucchini, tomato sauce, garlic, basil, parmesan. Ta

In [40]:
while True:
    user_query = input("\nAsk the food chatbot something, or type 'exit': ")

    if user_query.lower() == "exit":
        print("Goodbye!")
        break

    max_calories = input("Maximum calories? Press Enter to skip: ")
    diet_tag = input("Diet tag? vegetarian/vegan/high-protein/etc. Press Enter to skip: ")

    max_calories = int(max_calories) if max_calories.strip() else None
    diet_tag = diet_tag if diet_tag.strip() else None

    response = rag_food_chatbot_with_filters(
        user_query=user_query,
        max_calories=max_calories,
        diet_tag=diet_tag,
        n_results=5
    )

    print("\n" + response)


Ask the food chatbot something, or type 'exit': something light pls
Maximum calories? Press Enter to skip: 100
Diet tag? vegetarian/vegan/high-protein/etc. Press Enter to skip: vegan

Sorry, I couldn't find matching foods. Try relaxing your filters.

Ask the food chatbot something, or type 'exit': something light pls
Maximum calories? Press Enter to skip: 100
Diet tag? vegetarian/vegan/high-protein/etc. Press Enter to skip: 

Based on your request: 'something light pls', here are my recommendations:

1. Miso Soup
   Cuisine: Japanese
   Calories: 90
   Meal Type: soup
   Diet Tags: vegetarian, low-calorie
   Why it matches: Miso Soup. A light Japanese soup with miso, tofu, seaweed, and scallions. Ingredients: miso paste, tofu, seaweed, scallions, dashi. Taste: umami, light. Cooking method: simmered.



Ask the food chatbot something, or type 'exit': exit
Goodbye!


In [42]:
!pip install groq

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [48]:
from groq import Groq

# Replace this with your actual API key
GROQ_API_KEY = "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"

groq_client = Groq(api_key=GROQ_API_KEY)

In [49]:
# LLM powered RAG

def groq_rag_food_chatbot(user_query, cuisine=None, max_calories=None, meal_type=None, diet_tag=None, n_results=5):
    conditions = []

    if cuisine:
        conditions.append({"cuisine": cuisine})

    if max_calories:
        conditions.append({"calories": {"$lte": int(max_calories)}})

    if meal_type:
        conditions.append({"meal_type": meal_type})

    if len(conditions) == 1:
        where_filter = conditions[0]
    elif len(conditions) > 1:
        where_filter = {"$and": conditions}
    else:
        where_filter = None

    query_args = {
        "query_texts": [user_query],
        "n_results": 20
    }

    if where_filter:
        query_args["where"] = where_filter

    results = collection.query(**query_args)

    retrieved_foods = []

    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]

        if diet_tag:
            if diet_tag.lower() not in metadata["diet_tags"].lower():
                continue

        retrieved_foods.append({
            "name": metadata["name"],
            "cuisine": metadata["cuisine"],
            "calories": metadata["calories"],
            "meal_type": metadata["meal_type"],
            "diet_tags": metadata["diet_tags"],
            "description": doc
        })

        if len(retrieved_foods) == n_results:
            break

    if not retrieved_foods:
        return "Sorry, I couldn't find matching foods. Try relaxing your filters."

    context = ""
    for food in retrieved_foods:
        context += (
            f"Name: {food['name']}\n"
            f"Cuisine: {food['cuisine']}\n"
            f"Calories: {food['calories']}\n"
            f"Meal Type: {food['meal_type']}\n"
            f"Diet Tags: {food['diet_tags']}\n"
            f"Description: {food['description']}\n\n"
        )

    prompt = f"""
You are a helpful food recommendation chatbot.

User request:
{user_query}

Retrieved food options from the database:
{context}

Give a friendly, concise recommendation based ONLY on the retrieved food options.
Mention why each recommendation fits the user's request.
Do not invent foods that are not in the retrieved options.
"""

    completion = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a helpful food recommendation assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4,
        max_tokens=500
    )

    return completion.choices[0].message.content

In [50]:
print(
    groq_rag_food_chatbot(
        user_query="I want something light and vegetarian",
        max_calories=350,
        diet_tag="vegetarian",
        n_results=5
    )
)

Based on your request for something light and vegetarian, I highly recommend the following options:

1. **Couscous Salad**: This Moroccan salad is a perfect fit, with only 290 calories and a fresh, herby taste. It's also a great combination of vegetables and herbs, making it a light and satisfying option.

2. **Hot and Sour Soup**: This Chinese soup is another excellent choice, with a tangy and peppery flavor that's both refreshing and light. With only 150 calories, it's a great option for a light meal or snack.

3. **Papaya Salad**: This Thai salad is a great choice for a light and refreshing meal. With a spicy and tangy taste, it's a perfect option for those looking for something exciting and easy to digest.

These options are all vegetarian, low in calories, and offer a variety of flavors and textures to choose from. Enjoy!


In [ ]:
while True:
    user_query = input("\n🍽️ Ask the food chatbot something, or type 'exit': ")

    if user_query.lower().strip() == "exit":
        print("Goodbye!")
        break

    cuisine = input("🌍 Cuisine? Press Enter to skip: ")
    max_calories = input("🔥 Maximum calories? Press Enter to skip: ")
    meal_type = input("🍱 Meal type? Press Enter to skip: ")
    diet_tag = input("🥗 Diet tag? vegetarian/vegan/high-protein/etc. Press Enter to skip: ")

    cuisine = cuisine.strip() if cuisine.strip() else None
    max_calories = int(max_calories) if max_calories.strip() else None
    meal_type = meal_type.strip() if meal_type.strip() else None
    diet_tag = diet_tag.strip() if diet_tag.strip() else None

    response = groq_rag_food_chatbot(
        user_query=user_query,
        cuisine=cuisine,
        max_calories=max_calories,
        meal_type=meal_type,
        diet_tag=diet_tag,
        n_results=5
    )

    print("\n🤖 Food Chatbot Response:\n")
    print(response)


🍽️ Ask the food chatbot something, or type 'exit': im carving something light but spicy
🌍 Cuisine? Press Enter to skip: 
🔥 Maximum calories? Press Enter to skip: 
🍱 Meal type? Press Enter to skip: 
🥗 Diet tag? vegetarian/vegan/high-protein/etc. Press Enter to skip: 

🤖 Food Chatbot Response:

Based on your request for something light but spicy, I recommend the following options:

1. **Paneer Tikka**: This Indian appetizer is light, with only 320 calories, and packs a spicy punch. The marinated paneer and vegetables are grilled to perfection, giving it a smoky flavor.
2. **Chana Masala**: While it's a main dish, Chana Masala is relatively light with 380 calories. It's a spicy and savory chickpea curry that's perfect for those who want a flavorful meal without feeling too full.

Both of these options should satisfy your request for something light but spicy.

🍽️ Ask the food chatbot something, or type 'exit': something light
🌍 Cuisine? Press Enter to skip: japanese
🔥 Maximum calories? P